# Aufgabe 2: CNN zur Auto-Erkennung

Zur Erkennung von mehreren Autos auf Bildern werden 2 Ansätze verfolgt:
1. Verwendung des vortrainierten Netzes mit einem Sliding Window
2. Verwendung des vortrainierten Netzes mit Detection-Algorithmus YOLOv8

# Inhaltsverzeichnis
[1 Allgemeiner Aufbau](#allgemeiner-aufbau)  
&nbsp;&nbsp;&nbsp;&nbsp;[1.1 Vorbereitung des Models und der Funktionen](#11-vorbereitung-des-models-und-der-funktionen)  
&nbsp;&nbsp;&nbsp;&nbsp;[1.2 Zusammenstellung des kompletten Algorithmus](#zusammenstellung-des-kompletten-algorithmus)  
[2 Bilderkennung bei den gegebenen Bildern](#2-bilderkennung-bei-den-gegebenen-bildern)  
&nbsp;&nbsp;&nbsp;&nbsp;[2.1 Gegebenes Bild 1](#21-gegebenes-bild-1)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[2.1.1 Detektion in gegebenem Bild 1](#211-detektion-in-gegebenem-bild-1)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[2.1.2 Bewertung der Objekterkennung in gegebenem Bild 1](#212-bewertung-der-objekterkennung-in-gegebenem-bild-1)  
&nbsp;&nbsp;&nbsp;&nbsp;[2.2 Gegebenes Bild 2](#22-gegebenes-bild-2)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[2.2.1 Detektion in gegebenem Bild 2](#221-detektion-in-gegebenem-bild-2)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[2.2.2 Bewertung der Objekterkennung in gegebenem Bild 2](#222-bewertung-der-objekterkennung-in-gegebenem-bild-2)  
&nbsp;&nbsp;&nbsp;&nbsp;[2.3 Gegebenes Bild 3](#23-gegebenes-bild-3)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[2.3.1 Detektion in gegebenem Bild 3](#231-detektion-in-gegebenem-bild-3)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[2.3.2 Bewertung der Objekterkennung in gegebenem Bild 3](#232-bewertung-der-objekterkennung-in-gegebenem-bild-3)  
[3 Internetbilder](#3-internetbilder)  
&nbsp;&nbsp;&nbsp;&nbsp;[3.1 Internetbild 1](#31-internetbild-1)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[3.1.1 Detektion in Internetbild 1](#311-detektion-in-internetbild-1)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[3.1.2 Bewertung der Objekterkennung in Internetbild 1](#312-bewertung-der-objekterkennung-in-internetbild-1)  
&nbsp;&nbsp;&nbsp;&nbsp;[3.2 Internetbild 2](#32-internetbild-2)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[3.2.1 Detektion in Internetbild 2](#321-detektion-in-internetbild-2)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[3.2.2 Bewertung der Objekterkennung in Internetbild 2](#322-bewertung-der-objekterkennung-in-internetbild-2)  
&nbsp;&nbsp;&nbsp;&nbsp;[3.3 Internetbild 3](#33-internetbild-3)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[3.3.1 Detektion in Internetbild 3](#331-detektion-in-internetbild-3)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[3.3.2 Bewertung der Objekterkennung in Internetbild 3](#332-bewertung-der-objekterkennung-in-internetbild-3)  


# 1 Allgemeiner Aufbau

Zur Erkennung von Autos in Bildern mittels des vortrainierten Netzes werden folgende Ansätze/Funktionen verwendet:
- Sliding Window
- Vorverarbeitung und Batchverarbeitung von Bildausschnitten
- Non Maximum Surpression (NMS)
- Image Pyramid
- Entfernung verschachtelter Bounding Boxes

Diese Ansätze/Funktionen sind in den nachfolgenden Funktionen implementiert und in deren Docstrings genauer beschrieben.

Als weitere Hilfsfunktion wird `put_bounding_boxes_and_save` zur Visualisierung und Speicherung der Ergebnisse genutzt.

Die Zusammenführung der einzelnen Funktionen erfolgt in `complete_detection_algorithm`.

Weiterhin wurde für die Parameterauswahl bei den nachfolgenden Bilderkennungen eine feste Kopie des Keras-Modells aus Aufgabe 1a zum Stand 15.04.2026 verwendet. Dies soll sicherstellen, dass ein etwaiges Nachtrainieren oder Experementieren bei den Modelltrainings nicht zu direkten (verschlechternden) Auswirkungen auf die Funktionalität der Bilderkennungsparameter in dieser Aufgabe führt.

---

## 1.1 Vorbereitung des Models und der Funktionen

In [1]:
# Imports

from tensorflow import keras
from pathlib import Path
import numpy as np
import cv2

import random


In [2]:
# Laden des Models

current_dir = Path.cwd()
model_path = current_dir.parent / "models" / "task1a" / "hardcopy_car_cnn_15042026.keras"
model = keras.models.load_model(model_path)

In [3]:
# sliding window
def sliding_window(image, step_size, window_size):
    """
    Iteriert über ein Bild mittels Sliding Window-Verfahren.
    
    Args:
        image: Eingabebild
        step_size: Schrittweite in Pixeln
        window_size: Tuple (Breite, Höhe) des Fensters
        
    Yields:
        Tuple (x, y, window): Position und Bildausschnitt
    """
    window_width = window_size[0]
    window_height = window_size[1]
    image_height, image_width = image.shape[:2]

    # Iteration über Window
    for y in range (0, image_height, step_size):
        for x in range (0, image_width, step_size):
            window = image[y : y + window_height, x : x + window_width]
            
            if (window.shape[:2] != (window_height, window_width)) :
                continue
            yield (x, y, window)


def preprocess_window(window):
    """
    Bereitet Bildausschnitt für CIFAR-10-Modell vor.
    
    Skaliert auf 32x32 Pixel, konvertiert zu RGB und normalisiert auf [0, 1].
    
    Args:
        window: Bildausschnitt (BGR)
        
    Returns:
        Normalisiertes RGB-Bild als float32
    """

    if window.shape[:2] != (32, 32):
        window = cv2.resize(window, (32, 32), interpolation = cv2.INTER_AREA)
    
    window_rgb = cv2.cvtColor(window, cv2.COLOR_BGR2RGB)
    window_normalized = window_rgb.astype(np.float32) / 255.0
    
    return window_normalized


In [4]:
# sliding window batch
def sliding_window_batch(image, step_size, window_size, model, threshold):
    """
    Führt Sliding Window-Detektion mit Batch-Verarbeitung durch.
    
    Extrahiert alle Fenster aus dem Bild, klassifiziert diese gebündelt
    und gibt Detektionen zurück, die den Konfidenz-Schwellwert überschreiten.
    
    Args:
        image: Eingabebild
        step_size: Schrittweite in Pixeln
        window_size: Tuple (Breite, Höhe) des Fensters
        model: Keras-Modell für die Klassifikation
        threshold: Minimale Konfidenz für positive Detektionen
        
    Returns:
        List von Detektion-Dictionaries mit Schlüsseln:
        'x_start', 'y_start', 'x_end', 'y_end', 'confidence'
    """    
    window_width = window_size[0]
    window_height = window_size[1]
    image_height, image_width = image.shape[:2]
    windows = []
    positions = []

   # Auswahl Windows
    for y in range (0, image_height, step_size):
        for x in range (0, image_width, step_size):
            window = image[y : y + window_height, x : x + window_width]
            
            if (window.shape[:2] == (window_height, window_width)) :
                windows.append(preprocess_window(window))
                positions.append((x, y))
                            
    # Batch Vorhersage
    windows_array = np.array(windows)
    predictions = model.predict(windows_array, batch_size=64, verbose=0)

    # Relevante Ergebnisse sammeln
    detections = []
    for (x, y), pred in zip(positions, predictions):
        confidence = pred[0]
        if confidence > threshold:
            detections.append({
                'x_start': x, 
                'y_start': y, 
                'x_end': x + window_width,
                'y_end': y + window_height,
                'confidence': float(confidence)
            })
    
    return detections



In [5]:
# Non maximum surpression

def compute_iou(box1, box2):
    """
    Berechnet Intersection over Union (IoU) zwischen zwei Bounding Boxes.
    
    Args:
        box1: Dictionary mit Schlüsseln 'x_start', 'y_start', 'x_end', 'y_end'
        box2: Dictionary mit Schlüsseln 'x_start', 'y_start', 'x_end', 'y_end'
        
    Returns:
        IoU-Wert zwischen 0 (keine Überlappung) und 1 (identisch)
    """
    # Überlappungsbereich
    x1 = max(box1['x_start'], box2['x_start'])
    y1 = max(box1['y_start'], box2['y_start'])
    x2 = min(box1['x_end'], box2['x_end'])
    y2 = min(box1['y_end'], box2['y_end'])
    
    # Keine Überlappung
    if x2 < x1 or y2 < y1:
        return 0.0
    
    # Flächen
    intersection = (x2 - x1) * (y2 - y1)
    area1 = (box1['x_end'] - box1['x_start']) * (box1['y_end'] - box1['y_start'])
    area2 = (box2['x_end'] - box2['x_start']) * (box2['y_end'] - box2['y_start'])
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0

def non_max_suppression(detections, overlap_thresh=0.3):
    """
    Entfernt überlappende Detektionen mittels Non-Maximum Suppression.
    
    Behält jeweils die Detektion mit höchster Konfidenz und unterdrückt
    alle überlappenden Detektionen, deren IoU den Schwellwert überschreitet.
    
    Args:
        detections: Liste von Detektions-Dictionaries mit 'confidence'
        overlap_thresh: Maximaler IoU-Wert für tolerierte Überlappung (Standard: 0.3)
        
    Returns:
        Gefilterte Liste von Detektionen ohne signifikante Überlappungen
    """
    if len(detections) == 0:
        return []
    
    # Nach Confidence sortieren (höchste zuerst)
    detections = sorted(detections, key=lambda x: x['confidence'], reverse=True)
    keep = []
    
    while len(detections) > 0:
        # Beste Detection übernehmen
        best = detections.pop(0)
        keep.append(best)
        
        # Überlappende Detections entfernen
        detections = [
            det for det in detections 
            if compute_iou(best, det) < overlap_thresh
        ]
    return keep

In [6]:
# image pyramid

def image_pyramid (image, scales = [1, 0.8, 0.6, 0.4, 0.2, 0.1, 0.05], window_size = 32):
    """
    Erstellt eine Bildpyramide mit mehreren Skalierungsstufen.

    Überspringt Skalen, die kleiner als die Fenstergröße werden.
    
    Args:
        image: Eingabebild
        scales: Liste der Skalierungsfaktoren (Standard: [1, 0.8, 0.6, 0.4, 0.2, 0.1, 0.05])
        window_size: Minimale Bildgröße (Breite, Höhe) oder einzelner Wert
        
    Returns:
        Liste von Tupeln (skaliertes_bild, skalierungsfaktor)
    """
    pyramid = []
    
    for scale in scales:
        new_width = int(image.shape[1] * scale)
        new_height = int(image.shape[0] * scale)
        
        # Überspringe zu kleine Bilder
        if new_width < window_size[0] or new_height < window_size[1]:
            continue
            
        scaled = cv2.resize(image, (new_width, new_height))
        pyramid.append((scaled, scale))
    
    return pyramid

In [7]:
# Entfernen von nested boxes

def is_box_contained(small_box, large_box, thresh=0.7):
    """
    Prüft ob eine Box zu einem definierten Anteil in einer anderen enthalten ist.
    
    Args:
        small_box: Zu prüfende Box (Dictionary mit Koordinaten)
        large_box: Umgebende Box (Dictionary mit Koordinaten)
        thresh: Schwellwert für Containment-Ratio (Standard: 0.7)
        
    Returns:
        True wenn small_box zu >thresh% in large_box liegt, sonst False
    """
    x1 = max(small_box['x_start'], large_box['x_start'])
    y1 = max(small_box['y_start'], large_box['y_start'])
    x2 = min(small_box['x_end'], large_box['x_end'])
    y2 = min(small_box['y_end'], large_box['y_end'])
    
    if x2 < x1 or y2 < y1:
        return False
    
    small_area = (small_box['x_end'] - small_box['x_start']) * \
                 (small_box['y_end'] - small_box['y_start'])
    
    intersection = (x2 - x1) * (y2 - y1)
    containment_ratio = intersection / small_area if small_area > 0 else 0
    
    return containment_ratio > thresh


def remove_contained_boxes(detections, containment_thresh=0.7):
    """
    Entfernt Bounding Boxes, die in größeren Boxen enthalten sind.
        
    Args:
        detections: Liste von Detektions-Dictionaries
        containment_thresh: Schwellwert für Containment-Ratio (Standard: 0.7)
        
    Returns:
        Gefilterte Liste ohne verschachtelte Detektionen
    """
    if len(detections) == 0:
        return []
        
    # Nach Fläche sortieren (größte zuerst)
    detections = sorted(detections, 
                       key=lambda x: (x['x_end'] - x['x_start']) * (x['y_end'] - x['y_start']), 
                       reverse=True)
    
    keep = []
    
    for box in detections:
        is_contained = False
        
        # Prüfe gegen bereits beibehaltene (größere) Boxen
        for kept_box in keep:
            if is_box_contained(box, kept_box, containment_thresh):
                is_contained = True
                break
        
        if not is_contained:
            keep.append(box)
    
    return keep

In [8]:
# Hilfsfunktionen für Visualisierung und Speicherung

def put_bounding_boxes_and_save(image, detections, im_origin, im_no):
    """
    Zeichnet Bounding Boxes auf das Bild und speichert Ergebnisse.
    
    Args:
        image: Eingabebild
        detections: Liste von Detektions-Dictionaries mit Koordinaten und Konfidenz
        im_origin: Herkunftsbezeichnung für Dateinamen (z.B. "test_", "train_")
        im_no: Bildnummer für Dateinamen
        
    Returns:
        Bild mit allen eingezeichneten Bounding Boxes
        
    Side Effects:
        Speichert Bilder in ../src/result_images/:
        - result_{im_origin}bild{im_no}_det_{idx}.jpg (Einzeldetektionen)
        - result_{im_origin}bild{im_no}_total.jpg (Gesamtbild)
    """    
    result = image.copy()
    font_size = round(image.shape[0] / 600, 1)
    font_thickness = int(image.shape[1] / 350) 
        
    for idx, det in enumerate(detections):
        single_result = image.copy()
        color_bgr = (random.randint(150, 250), random.randint(0, 100), random.randint(100, 250))
        # Bounding Box
        cv2.rectangle(result, 
                    (det['x_start'], det['y_start']), 
                    (det['x_end'], det['y_end']), 
                    color_bgr, font_thickness)
        
        cv2.rectangle(single_result, 
            (det['x_start'], det['y_start']), 
            (det['x_end'], det['y_end']), 
            color_bgr, font_thickness)
        
        # Label mit Confidence
        label = f"Det. {idx+1} Conf. {det['confidence']:.2f}"
        cv2.putText(result, label, 
                    (det['x_start'], det['y_start'] - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, font_size, color_bgr, font_thickness)
        cv2.putText(single_result, label, 
                    (det['x_start'], det['y_start'] - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, font_size, color_bgr, font_thickness) 

        # Speichern Einzelbild
        cv2.imwrite(f"../src/result_images/result_{im_origin}bild{im_no}_det_{idx+1}.jpg", single_result)     

    # Speichern Gesamtbild    
    cv2.imwrite(f"../src/result_images/result_{im_origin}bild{im_no}_total.jpg", result)

    return result

## 1.2 Zusammenstellung des kompletten Algorithmus

Im folgenden Codeabschnitt wird die Auto-Detektion anhand den Aufrufen der obigen Funktionen orchestriert und als Funktion `complete_detection_algorithm` bereitgestellt.  
Die Ergebnisse der nachfolgenden Bildanalysen werden im Projektordner lernverfahren/src/result_images gespeichert.

In [9]:
# Kompleter Algorithmus, der Bilddaten und Parameter entgegen nimmt

def complete_detection_algorithm(step_size: int, 
                                 ws: list, 
                                 conf_th: float, 
                                 nms_th: float, 
                                 cnt_th: float, 
                                 scales: list, 
                                 im_origin: str, 
                                 im_no: int):
    """
    Führt vollständige Multi-Scale-Objektdetektion mit Sliding Window durch.
    
    Pipeline:
    1. Bild einlesen
    2. Bildpyramide erstellen (mehrere Skalierungen)
    3. Sliding Window-Detektion auf allen Skalen und Fenstergrößen
    4. Koordinaten auf Originalgröße zurückskalieren
    5. Non-Maximum Suppression zur Reduzierung von Überlappungen
    6. Entfernung verschachtelter Bounding Boxes
    7. Visualisierung und Speicherung der Ergebnisse
    
    Args:
        step_size: Schrittweite des Sliding Windows in Pixeln
        ws: Liste von Fenstergrößen als Tupel [(Breite, Höhe), ...]
        conf_th: Konfidenz-Schwellwert für positive Detektionen (0-1)
        nms_th: IoU-Schwellwert für Non-Maximum Suppression (0-1)
        cnt_th: Schwellwert für Containment-Ratio bei verschachtelten Boxen (0-1)
        scales: Liste der Skalierungsfaktoren für Bildpyramide
        im_origin: Präfix für Bilddateinamen (z.B. "", "in", "test")
        im_no: Bildnummer für Dateinamen
        
    Side Effects:
        - Lädt Bild aus ../src/input_images/{im_origin}bild{im_no}.jpg
        - Speichert Ergebnisbilder in ../src/result_images/
        - Gibt Detektionsstatistiken auf Konsole aus
        - Zeigt Ergebnisbild in OpenCV-Fenster an
    """    
    #=== PARAMETER ===
    STEP_SIZE = step_size               # Schrittweite Sliding Window
    WINDOW_SIZE = ws                    # Größe Sliding Window
    CONFIDENCE_THRESHOLD = conf_th      # Confidence Wert zur Erkennung eines Autos
    NMS_THRESHOLD = nms_th              # Maximale erlaubte Überlappung (IoU)
    CONTAINT_THRESHOLD = cnt_th         # Wert, ab dem kleinere Box als verschachtelt gilt und entfernt wird
    SCALES = scales                     # Skalierungsfaktoren

    IMAGE_ORIGIN = im_origin            # "" bei gegebenem Bild, "in" bei Bild aus Internet
    IMAGE_NO = im_no

    # 1. Bild einlesen
    image = cv2.imread(f"../src/input_images/{IMAGE_ORIGIN}bild{IMAGE_NO}.jpg")
    all_detections = []
    for window_size in WINDOW_SIZE:
        # 2. Erstellen skalierter Bilder über Image Pyramide
        scaled_images = image_pyramid(image, window_size=window_size, scales=SCALES)

        # 3. Über Skalen iterien
        for scaled_im, scale_factor in scaled_images:
            print(f"Imagesize: {scaled_im.shape[0]} und {scaled_im.shape[1]}")

            detections = sliding_window_batch(
                scaled_im, 
                step_size=STEP_SIZE, 
                window_size=window_size, 
                model=model, 
                threshold=CONFIDENCE_THRESHOLD)
            
            print(f"  Window {window_size}: {len(detections)} Detektionen")
                
            # Koordinaten zurück auf Original skalieren
            for det in detections:
                det['x_start'] = int(det['x_start'] / scale_factor)
                det['y_start'] = int(det['y_start'] / scale_factor)
                det['x_end'] = int(det['x_end'] / scale_factor)
                det['y_end'] = int(det['y_end'] / scale_factor)
                det['scale'] = scale_factor
                det['window_size'] = window_size

            all_detections.extend(detections)

    print(f"\nGesamt vor NMS: {len(all_detections)} Detektionen")

    # 4. NMS über alle Detektionen
    detections_filtered = non_max_suppression(
        all_detections, 
        overlap_thresh=NMS_THRESHOLD)
    print(f"\nGesamt nach NMS: {len(detections_filtered)} Detektionen")

    # 5. Entfernung verschachtelter Boxen
    detections_filtered = remove_contained_boxes(
        detections_filtered, 
        containment_thresh=CONTAINT_THRESHOLD)
    print(f"\nGesamt nach Entfernung Verschachtelung: {len(detections_filtered)} Detektionen")

    # 6. Visualisierung
    result = image.copy()

    for idx, det in enumerate(detections_filtered):
        print (f"Detektion {idx+1}: Confidence {det['confidence']:.2f} " 
            f"bei Skalierung @{det['scale']:.1f}x " 
            f"und Fenstergröße {det['window_size']}")
            
    result = put_bounding_boxes_and_save(
        image=result, 
        detections=detections_filtered, 
        im_origin=IMAGE_ORIGIN,
        im_no=IMAGE_NO)
    
    resize_factor = 1000/ result.shape[1]

    resized_result = cv2.resize(
        result, 
        (int(result.shape[1] * resize_factor), int(result.shape[0] * resize_factor)))

    cv2.imshow("Car Detection", resized_result)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

# 2 Bilderkennung bei den gegebenen Bildern

## 2.1 Gegebenes Bild 1

### 2.1.1 Detektion in gegebenem Bild 1

Für das Bild 1 haben sich bei den Parametertests folgende Einstellungen als am präzisesten herausgestellt:

``STEP_SIZE`` = 20  
``WINDOW_SIZE`` = [(128, 128), (256,256)]  
``CONFIDENCE_THRESHOLD`` = 0.95  
``NMS_THRESHOLD`` = 0.3  
``CONTAINT_THRESHOLD`` = 0.6  
``SCALES`` = [0.1, 0.2]

In [10]:
# Bild 1

#=== PARAMETER ===
STEP_SIZE = 20
WINDOW_SIZE = [(128, 128), (256,256)]
CONFIDENCE_THRESHOLD = 0.95
NMS_THRESHOLD = 0.3
CONTAINT_THRESHOLD = 0.6
SCALES = [0.1, 0.2]

IMAGE_ORIGIN = ""
IMAGE_NO = 1

complete_detection_algorithm(
    step_size=STEP_SIZE,
    ws=WINDOW_SIZE,
    conf_th=CONFIDENCE_THRESHOLD,
    nms_th=NMS_THRESHOLD,
    cnt_th=CONTAINT_THRESHOLD,
    scales=SCALES,
    im_origin=IMAGE_ORIGIN,
    im_no=IMAGE_NO    
)

Imagesize: 172 und 230
  Window (128, 128): 1 Detektionen
Imagesize: 345 und 460
  Window (128, 128): 1 Detektionen
Imagesize: 345 und 460
  Window (256, 256): 1 Detektionen

Gesamt vor NMS: 3 Detektionen

Gesamt nach NMS: 2 Detektionen

Gesamt nach Entfernung Verschachtelung: 1 Detektionen
Detektion 1: Confidence 0.96 bei Skalierung @0.1x und Fenstergröße (128, 128)


### 2.1.2 Bewertung der Objekterkennung in gegebenem Bild 1
Das einzige rote Auto im Bild wird optimal erkannt.  


---

## 2.2 Gegebenes Bild 2

### 2.2.1 Detektion in gegebenem Bild 2

Für das Bild 2 haben sich bei den Parametertests folgende Einstellungen als am präzisesten herausgestellt:

``STEP_SIZE`` = 20  
``WINDOW_SIZE`` = [(128, 128), (256,256)]  
``CONFIDENCE_THRESHOLD`` = 0.95  
``NMS_THRESHOLD`` = 0.3  
``CONTAINT_THRESHOLD`` = 0.6  
``SCALES`` = [0.4, 0.5]

In [11]:
# Bild 2

#=== PARAMETER ===
STEP_SIZE = 20
WINDOW_SIZE = [(128, 128), (256,256)]
CONFIDENCE_THRESHOLD = 0.95
NMS_THRESHOLD = 0.3
CONTAINT_THRESHOLD = 0.6
SCALES = [0.4, 0.5]

IMAGE_ORIGIN = ""
IMAGE_NO = 2

complete_detection_algorithm(
    step_size=STEP_SIZE,
    ws=WINDOW_SIZE,
    conf_th=CONFIDENCE_THRESHOLD,
    nms_th=NMS_THRESHOLD,
    cnt_th=CONTAINT_THRESHOLD,
    scales=SCALES,
    im_origin=IMAGE_ORIGIN,
    im_no=IMAGE_NO    
)

Imagesize: 864 und 1296
  Window (128, 128): 33 Detektionen
Imagesize: 1080 und 1620
  Window (128, 128): 29 Detektionen
Imagesize: 864 und 1296
  Window (256, 256): 46 Detektionen
Imagesize: 1080 und 1620
  Window (256, 256): 54 Detektionen

Gesamt vor NMS: 162 Detektionen

Gesamt nach NMS: 12 Detektionen

Gesamt nach Entfernung Verschachtelung: 8 Detektionen
Detektion 1: Confidence 0.97 bei Skalierung @0.4x und Fenstergröße (256, 256)
Detektion 2: Confidence 0.97 bei Skalierung @0.4x und Fenstergröße (256, 256)
Detektion 3: Confidence 0.97 bei Skalierung @0.4x und Fenstergröße (256, 256)
Detektion 4: Confidence 1.00 bei Skalierung @0.5x und Fenstergröße (256, 256)
Detektion 5: Confidence 1.00 bei Skalierung @0.5x und Fenstergröße (256, 256)
Detektion 6: Confidence 0.99 bei Skalierung @0.5x und Fenstergröße (256, 256)
Detektion 7: Confidence 1.00 bei Skalierung @0.4x und Fenstergröße (128, 128)
Detektion 8: Confidence 0.99 bei Skalierung @0.5x und Fenstergröße (128, 128)


### 2.2.2 Bewertung der Objekterkennung in gegebenem Bild 2

Die vorliegende Analyse der Objekterkennung zeigt eine überwiegend präzise Leistung bei der Identifizierung der Fahrzeuge im Bild.

**1. Korrekte Erkennungen (True Positives):**
*   Die farbigen Fahrzeuge (gelb, pink, grün, rot) werden optimal erkannt und mit exakten Bounding Boxes umschlossen.

**2. Abweichungen bei der Bounding Box:**
*   Das linke graue und schwarze Auto weisen eine leicht nach links verschobene Bounding Box auf.
*   **Analyse der Abweichung:** Eine detaillierte Untersuchung hat ergeben, dass die Verringerung der `Stepsize` – ein Parameter zur Feinjustierung der Rahmenbestimmung – zwar zu einer potenziell genaueren Positionierung dieser speziellen Bounding Boxen führen könnte. Dies führte jedoch zu einer signifikanten Zunahme von Fehlklassifikationen (z.B. False Positives oder schlechtere Treffer bei anderen Objekten) im Gesamtbild.
*   **Begründung der Akzeptanz:** Die aktuelle Positionierung der Bounding Box für das linke graue Auto wurde daher als bestmöglicher Kompromiss im Rahmen der gewählten Parameterkonfiguration akzeptiert, um eine höhere Gesamtqualität der Erkennung zu gewährleisten.

**3. Fehlklassifikationen (False Positives):**
*   Ein Rahmen zwischen dem grauen, pinken und grünen Auto wird fälschlicherweise als Objekt identifiziert.
*   **Analyse der Fehlklassifikation:** Dieses spezifische False Positive trat bei den meisten der getesteten Parameterkombinationen auf.
*   **Begründung der Akzeptanz:** Versuche, diese Fehlklassifikation zu eliminieren, führten zu einer deutlich schlechteren Gesamtqualität der Objekterkennung (z.B. weitere False Positives oder die Nichterkennung tatsächlicher Objekte). Aus diesem Grund wurde diese einzelne Fehlklassifikation als unvermeidbar im Kontext der optimierten Gesamtleistung akzeptiert.

**Fazit:**
Die Objekterkennung liefert in der vorliegenden Konfiguration insgesamt gute Ergebnisse, insbesondere bei den farbigen Fahrzeugen. Kleinere Abweichungen bei der Bounding Box des linken grauen Autos und eine spezifische Fehlklassifikation wurden nach umfassender Parameteroptimierung als notwendige Kompromisse für die Erzielung der bestmöglichen Gesamtperformance akzeptiert.

---

## 2.3 Gegebenes Bild 3

### 2.3.1 Detektion in gegebenem Bild 3

Für das Bild 3 haben sich bei den Parametertests folgende Einstellungen als am präzisesten herausgestellt:

``STEP_SIZE`` = 16  
``WINDOW_SIZE`` = [(96, 96), (128, 128)]  
``CONFIDENCE_THRESHOLD`` = 0.91  
``NMS_THRESHOLD`` = 0.3  
``CONTAINT_THRESHOLD`` = 0.6  
``SCALES`` = [0.2]


In [12]:
# Bild 3

#=== PARAMETER ===
STEP_SIZE = 16
WINDOW_SIZE = [(96, 96),(128, 128)]
CONFIDENCE_THRESHOLD = 0.91
NMS_THRESHOLD = 0.3
CONTAINT_THRESHOLD = 0.6
SCALES = [0.2]

IMAGE_ORIGIN = ""
IMAGE_NO = 3

complete_detection_algorithm(
    step_size=STEP_SIZE,
    ws=WINDOW_SIZE,
    conf_th=CONFIDENCE_THRESHOLD,
    nms_th=NMS_THRESHOLD,
    cnt_th=CONTAINT_THRESHOLD,
    scales=SCALES,
    im_origin=IMAGE_ORIGIN,
    im_no=IMAGE_NO    
)

Imagesize: 364 und 547
  Window (96, 96): 9 Detektionen
Imagesize: 364 und 547
  Window (128, 128): 10 Detektionen

Gesamt vor NMS: 19 Detektionen

Gesamt nach NMS: 4 Detektionen

Gesamt nach Entfernung Verschachtelung: 4 Detektionen
Detektion 1: Confidence 0.99 bei Skalierung @0.2x und Fenstergröße (128, 128)
Detektion 2: Confidence 0.95 bei Skalierung @0.2x und Fenstergröße (128, 128)
Detektion 3: Confidence 0.98 bei Skalierung @0.2x und Fenstergröße (96, 96)
Detektion 4: Confidence 0.94 bei Skalierung @0.2x und Fenstergröße (96, 96)


### 2.3.2 Bewertung der Objekterkennung in gegebenem Bild 3

Die vorliegende Analyse der Objekterkennung zeigt eine überwiegend präzise Leistung bei der Identifizierung der Fahrzeuge im Vordergrund des Bildes.

**1. Korrekte Erkennungen (True Positives):**
*   Die vier "Hauptautos" im Vordergrund werden durch die gewählte Parameterkonfiguration korrekt und zuverlässig erkannt.

**2. Fehlklassifikationen (False Positives):**
*   Es sind keine False Positives im Ergebnis der Objekterkennung feststellbar. Die Minimierung von Fehlklassifikationen war unser primäres Ziel der Parameteroptimierung.

**3. Nichterkennung von Objekten (False Negatives):**
*   Die im Hintergrund des Bildes befindlichen Autos können mit der aktuellen Parameterwahl nicht erkannt werden.
*   **Analyse der Nichterkennung:** Es wurde evaluiert, ob eine Erkennung dieser Hintergrundobjekte grundsätzlich möglich ist. Als Beispiel wurde das weiße Auto im linken Hintergrund betrachtet. Durch eine Anpassung der Bildskalierung auf den Faktor 0.6 in Kombination mit einer Windowsize von (128, 128) war es möglich, dieses spezifische Fahrzeug zu erkennen.
*   **Begründung der Akzeptanz:** Diese alternative Parameterkombination führte jedoch zu einer signifikant hohen Anzahl an False Positives im gesamten Bild. Da die Reduzierung von False Positives im Fokus der Optimierung stand, wurde die aktuell festgelegte Parameterwahl als der beste Kompromiss zur Erzielung einer hohen Präzision bei gleichzeitiger Vermeidung von Fehlklassifikationen im Vordergrund akzeptiert.

**Fazit:**
Die gewählte Parameterkonfiguration bietet eine optimale Kompromisslösung, indem sie eine präzise Erkennung der relevanten Fahrzeuge im Vordergrund ohne False Positives ermöglicht. Die Nichterkennung von Hintergrundobjekten wird zugunsten der hohen Präzision und der Vermeidung von Fehlklassifikationen in Kauf genommen.

# 3 Internetbilder

## 3.1 Internetbild 1

### 3.1.1 Detektion in Internetbild 1

Für das Internetbild 1 haben sich bei den Parametertests folgende Einstellungen als am präzisesten herausgestellt:

``STEP_SIZE`` = 10  
`WINDOW_SIZE` = [(96, 96),(128, 128)]  
`CONFIDENCE_THRESHOLD` = 0.88   
`NMS_THRESHOLD` = 0.3  
`CONTAINT_THRESHOLD` = 0.6    
`SCALES` = [1.2, 1.5, 1.7, 2, 2.2, 2.4]  

In [13]:
# Internetbild 1

#=== PARAMETER ===
STEP_SIZE = 10
WINDOW_SIZE = [(96, 96),(128, 128)]
CONFIDENCE_THRESHOLD = 0.88
NMS_THRESHOLD = 0.3
CONTAINT_THRESHOLD = 0.6
SCALES = [1.2, 1.5, 1.7, 2, 2.2, 2.4]

IMAGE_ORIGIN = "in"
IMAGE_NO = 1
complete_detection_algorithm(
    step_size=STEP_SIZE,
    ws=WINDOW_SIZE,
    conf_th=CONFIDENCE_THRESHOLD,
    nms_th=NMS_THRESHOLD,
    cnt_th=CONTAINT_THRESHOLD,
    scales=SCALES,
    im_origin=IMAGE_ORIGIN,
    im_no=IMAGE_NO    
)

Imagesize: 516 und 1152
  Window (96, 96): 14 Detektionen
Imagesize: 645 und 1440
  Window (96, 96): 16 Detektionen
Imagesize: 731 und 1632
  Window (96, 96): 12 Detektionen
Imagesize: 860 und 1920
  Window (96, 96): 7 Detektionen
Imagesize: 946 und 2112
  Window (96, 96): 3 Detektionen
Imagesize: 1032 und 2304
  Window (96, 96): 11 Detektionen
Imagesize: 516 und 1152
  Window (128, 128): 7 Detektionen
Imagesize: 645 und 1440
  Window (128, 128): 24 Detektionen
Imagesize: 731 und 1632
  Window (128, 128): 26 Detektionen
Imagesize: 860 und 1920
  Window (128, 128): 26 Detektionen
Imagesize: 946 und 2112
  Window (128, 128): 26 Detektionen
Imagesize: 1032 und 2304
  Window (128, 128): 21 Detektionen

Gesamt vor NMS: 193 Detektionen

Gesamt nach NMS: 10 Detektionen

Gesamt nach Entfernung Verschachtelung: 8 Detektionen
Detektion 1: Confidence 1.00 bei Skalierung @1.5x und Fenstergröße (128, 128)
Detektion 2: Confidence 0.98 bei Skalierung @1.7x und Fenstergröße (128, 128)
Detektion 3: Con

### 3.1.2 Bewertung der Objekterkennung in Internetbild 1

**1. Korrekte Erkennungen (True Positives):**

Acht von neun primären Fahrzeuge im mittleren und unteren Vordergrund des Bildes werden präzise mit hohen Konfidenzwerten zwischen 0.95 und 1.00 erkannt und korrekt begrenzt.  

**2. Fehlklassifikationen (False Positives):**

Es sind keine False Positives erkennbar. Die aktuellen Parameter verhindern erfolgreich das Erkennen von irrelevanten Objekten oder Bildrauschen als Fahrzeuge.

**3. Nichterkennung von Objekten (False Negatives):**

Ein vollständiges Fahrzeug im unteren Bildbereich (weißes Auto) wird nicht erkannt.

*   **Analyse der Nichterkennung:** Dieses Fahrzeug wurde bei allen stabilen Parameterkombinationen ohne sonstige False Positives nicht erkannt.
*   **Begründung der Akzeptanz:** Die Fokussierung auf die Eliminierung von False Positives und die präzise Erkennung der prominentesten Fahrzeuge rechtfertigt die Nichterkennung dieses einzelnen Objekts. Eine Lockerung der Parameter zur Erfassung dieses Fahrzeugea hat in Tests zu einem Anstieg der Fehlklassifikationen geführt.

**Fazit:**
Die gewählte Konfiguration erreicht eine hohe Präzision und vermeidet Fehlklassifikationen im relevantesten Bildbereich. Die Nichterkennung wird bewusst in Kauf genommen, um die Robustheit und Fehlerfreiheit der Detektion zu gewährleisten.

---

## 3.2 Internetbild 2

### 3.2.1 Detektion in Internetbild 2

`STEP_SIZE` = 6  
`WINDOW_SIZE` = [(256, 256), (384, 384), (420, 420)]  
`CONFIDENCE_THRESHOLD` = 0.90  
`NMS_THRESHOLD` = 0.3  
`CONTAINT_THRESHOLD` = 0.5  
`SCALES` = [0.1, 0.2, 0.3]

In [14]:
# Internetbild 2

#=== PARAMETER ===
STEP_SIZE = 6
WINDOW_SIZE = [(256, 256), (384, 384), (420, 420)]
CONFIDENCE_THRESHOLD = 0.90
NMS_THRESHOLD = 0.3
CONTAINT_THRESHOLD = 0.5
SCALES = [0.1, 0.2, 0.3]

IMAGE_ORIGIN = "in"
IMAGE_NO = 2
complete_detection_algorithm(
    step_size=STEP_SIZE,
    ws=WINDOW_SIZE,
    conf_th=CONFIDENCE_THRESHOLD,
    nms_th=NMS_THRESHOLD,
    cnt_th=CONTAINT_THRESHOLD,
    scales=SCALES,
    im_origin=IMAGE_ORIGIN,
    im_no=IMAGE_NO    
)

Imagesize: 345 und 518
  Window (256, 256): 73 Detektionen
Imagesize: 691 und 1036
  Window (256, 256): 174 Detektionen
Imagesize: 1036 und 1555
  Window (256, 256): 32 Detektionen
Imagesize: 691 und 1036
  Window (384, 384): 446 Detektionen
Imagesize: 1036 und 1555
  Window (384, 384): 400 Detektionen
Imagesize: 691 und 1036
  Window (420, 420): 354 Detektionen
Imagesize: 1036 und 1555
  Window (420, 420): 578 Detektionen

Gesamt vor NMS: 2057 Detektionen

Gesamt nach NMS: 12 Detektionen

Gesamt nach Entfernung Verschachtelung: 6 Detektionen
Detektion 1: Confidence 0.98 bei Skalierung @0.1x und Fenstergröße (256, 256)
Detektion 2: Confidence 1.00 bei Skalierung @0.2x und Fenstergröße (420, 420)
Detektion 3: Confidence 0.98 bei Skalierung @0.3x und Fenstergröße (384, 384)
Detektion 4: Confidence 0.96 bei Skalierung @0.3x und Fenstergröße (384, 384)
Detektion 5: Confidence 0.98 bei Skalierung @0.3x und Fenstergröße (256, 256)
Detektion 6: Confidence 0.92 bei Skalierung @0.3x und Fenster

### 3.2.2 Bewertung der Objekterkennung in Internetbild 1

**1. Korrekte Erkennungen (True Positives):**

Sechs der sieben Hauptfahrzeuge im Vorder- und Mittelgrund werden korrekt und mit hohen Konfidenzwerten (zwischen 0.92 und 1.00) erkannt.

**2. Fehlklassifikationen (False Positives):**

Es sind keine False Positives im Ergebnis der Objekterkennung feststellbar.

**3. Nichterkennung von Objekten (False Negatives):**

Das dunkelblaue Auto ganz hinten rechts im Bild wird nicht erkannt. Es ist stark verdeckt und nur teilweise sichtbar.
*   **Analyse der Nichterkennung:** Die starke Verdeckung und möglicherweise die geringere Größe aufgrund der Perspektive führen dazu, dass dieses Objekt nicht detektiert wird.
*   **Begründung der Akzeptanz:**  Die Optimierung konzentriert sich auf eine hohe Präzision und die zuverlässige Erkennung der prominentesten und am besten sichtbaren Fahrzeuge. Eine Lockerung der Parameter (kleine Windowsizes und höhere Skalierungsfaktoren) zur Erfassung dieser Fahrzeuge hat in Tests zu einem Anstieg der Fehlklassifikationen geführt.

**Fazit:** 
Die gewählte Konfiguration liefert eine sehr gute, präzise Objekterkennung für die meisten prominenten Fahrzeuge im Bild, ohne dabei False Positives zu erzeugen. Die Nichterkennung des stark verdeckten Fahrzeugs im Hintergrund ist ein akzeptabler Kompromiss, um die hohe Präzision und Robustheit des Systems zu wahren.

---

## 3.3 Internetbild 3

### 3.3.1 Detektion in Internetbild 3

Für das Internetbild 3 haben sich bei den Parametertests folgende Einstellungen als am präzisesten herausgestellt:

`STEP_SIZE` = 10  
`WINDOW_SIZE` = [(96, 96),(128, 128)]  
``CONFIDENCE_THRESHOLD`` = 0.90  
`NMS_THRESHOLD` = 0.3  
`CONTAINT_THRESHOLD` = 0.5  
`SCALES` = [0.3, 0.5, 0.7, 1.5, 2.2]

In [16]:
# Internetbild 3

#=== PARAMETER ===
STEP_SIZE = 10
WINDOW_SIZE = [(96, 96),(128, 128)]
CONFIDENCE_THRESHOLD = 0.90
NMS_THRESHOLD = 0.3
CONTAINT_THRESHOLD = 0.5
SCALES = [0.3, 0.5, 0.7, 1.5, 2.2]

IMAGE_ORIGIN = "in"
IMAGE_NO = 3
complete_detection_algorithm(
    step_size=STEP_SIZE,
    ws=WINDOW_SIZE,
    conf_th=CONFIDENCE_THRESHOLD,
    nms_th=NMS_THRESHOLD,
    cnt_th=CONTAINT_THRESHOLD,
    scales=SCALES,
    im_origin=IMAGE_ORIGIN,
    im_no=IMAGE_NO    
)

Imagesize: 384 und 576
  Window (96, 96): 4 Detektionen
Imagesize: 640 und 960
  Window (96, 96): 6 Detektionen
Imagesize: 896 und 1344
  Window (96, 96): 1 Detektionen
Imagesize: 1920 und 2880
  Window (96, 96): 2 Detektionen
Imagesize: 2816 und 4224
  Window (96, 96): 3 Detektionen
Imagesize: 384 und 576
  Window (128, 128): 0 Detektionen
Imagesize: 640 und 960
  Window (128, 128): 12 Detektionen
Imagesize: 896 und 1344
  Window (128, 128): 9 Detektionen
Imagesize: 1920 und 2880
  Window (128, 128): 4 Detektionen
Imagesize: 2816 und 4224
  Window (128, 128): 3 Detektionen

Gesamt vor NMS: 44 Detektionen

Gesamt nach NMS: 9 Detektionen

Gesamt nach Entfernung Verschachtelung: 6 Detektionen
Detektion 1: Confidence 0.93 bei Skalierung @0.3x und Fenstergröße (96, 96)
Detektion 2: Confidence 0.99 bei Skalierung @0.5x und Fenstergröße (128, 128)
Detektion 3: Confidence 0.92 bei Skalierung @0.5x und Fenstergröße (128, 128)
Detektion 4: Confidence 0.97 bei Skalierung @0.7x und Fenstergröße (

### 3.3.2 Bewertung der Objekterkennung in Internetbild 1

**1. Korrekte Erkennungen (True Positives):**

Die sechs Fahrzeuge im Bild werden korrekt und mit hohen Konfidenzwerten (zwischen 0.91 und 1.00) erkannt. Die Bounding Boxes passen die Konturen der erkannten Objekte sehr gut an.

**2. Fehlklassifikationen (False Positives):**

Es sind keine False Positives im Ergebnis der Objekterkennung feststellbar.

**3. Nichterkennung von Objekten (False Negatives):**

Der weiße Lieferwagen im Bildmittelpunkt bleibt unerkannt, da seine Bounding Box aufgrund der räumlichen Überlappung mit dem links angrenzenden schwarzen Fahrzeug vom Algorithmus zusammengeführt wird. Dies stellt keine modellbasierte Fehlerkennung dar, sondern eine algorithmusbedingte Ungenauigkeit.

Diese Limitation wird toleriert, da die Eliminierung überlappender Bounding Boxes grundsätzlich ein effektives Mittel zur Selektion der präzisesten Detektionen darstellt.

**Fazit:**
Die gewählte Konfiguration liefert eine sehr gute, präzise Objekterkennung für die meisten prominenten Fahrzeuge im Bild, ohne dabei False Positives zu erzeugen. Die Nichterkennung des weißen Fahrzeuges wird wie beschrieben akzeptiert.